In [1]:
# Imports 

import random
from faker import Faker

fake = Faker("fr_FR")

Faker.seed(42)
random.seed(42)

In [2]:
# Test Faker 

for _ in range(3):
    date = fake.date_between(start_date="-1y", end_date="today")
    print(date.strftime("%d/%m"), "|", fake.city().upper(), "|", fake.random_number(digits=4, fix_len=True))

22/06 | SAINT NICOLE-LA-FORÊT | 5012
11/01 | GALLET | 2679
06/07 | FERNANDEZNEC | 1520


In [3]:
# First catalog of the merchants 

catalog = {
    "Groceries": [
        "CARREFOUR", "CARREFOUR MARKET", "LIDL", "AUCHAN", "E.LECLERC",
        "MONOPRIX", "FRANPRIX", "INTERMARCHE", "PICARD", "BIOCOOP",
    ],
    "Subscriptions": [
        "NETFLIX.COM", "SPOTIFY", "DISNEY PLUS", "CANAL+", "DEEZER",
        "AMAZON PRIME", "APPLE.COM/BILL", "YOUTUBE PREMIUM",
    ],
}

for category, merchants in catalog.items():
    print(f"{category}: {len(merchants)} merchants")

Groceries: 10 merchants
Subscriptions: 8 merchants


In [4]:
# Turning a merchant into a realistic label

def card_label(merchant):
    date = fake.date_between(start_date="-1y", end_date="today").strftime("%d/%m")
    city = fake.city().upper()
    card = f"CARTE X{fake.random_number(digits=4, fix_len=True)}"
    templates = [
        f"CB {merchant} {date}",
        f"CB {merchant} {city} {date}",
        f"CARTE {date} {merchant} {city}",
        f"CB {merchant} {date} {card}",
        f"PAIEMENT PAR CARTE {merchant} {date}",
    ]
    return random.choice(templates)


def direct_debit_label(merchant):
    ref = fake.bothify("FR##ZZZ######")
    templates = [
        f"PRLV SEPA {merchant}",
        f"PRLV SEPA {merchant} {ref}",
        f"PRELEVEMENT {merchant} REF {ref}",
    ]
    return random.choice(templates)


print("Card payments (Groceries):")
for merchant in random.sample(catalog["Groceries"], 4):
    print("  ", card_label(merchant))

print("\nDirect debits (Subscriptions):")
for merchant in random.sample(catalog["Subscriptions"], 4):
    print("  ", direct_debit_label(merchant))

Card payments (Groceries):
   CB CARREFOUR MARKET SAINT NICOLASVILLE 28/10
   CB CARREFOUR HUMBERT-SUR-MER 22/07
   CB E.LECLERC 02/06
   PAIEMENT PAR CARTE BIOCOOP 15/07

Direct debits (Subscriptions):
   PRLV SEPA SPOTIFY
   PRLV SEPA DEEZER
   PRLV SEPA CANAL+
   PRLV SEPA NETFLIX.COM


In [5]:
# Realistic noise helper

import unicodedata


def strip_accents(text):
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(c for c in normalized if not unicodedata.combining(c))


CITIES = [
    "PARIS", "PARIS 11", "PARIS 15", "LYON", "LYON 3", "MARSEILLE", "TOULOUSE",
    "NICE", "NANTES", "STRASBOURG", "MONTPELLIER", "BORDEAUX", "LILLE", "RENNES",
    "REIMS", "SAINT-DENIS", "BOULOGNE-BILL", "MONTREUIL", "NANTERRE", "VERSAILLES",
    "CRETEIL", "GRENOBLE", "DIJON", "ANGERS", "NIMES", "AIX-EN-PROVEN", "ROUEN",
    "TOULON", "LE HAVRE", "VILLEURBANNE", "ARGENTEUIL", "LA DEFENSE",
]


def random_date():
    return fake.date_between(start_date="-1y", end_date="today").strftime("%d/%m")


def random_city():
    return random.choice(CITIES)


def random_card():
    return f"CARTE X{fake.random_number(digits=4, fix_len=True)}"


def random_ref():
    return fake.bothify("FR##ZZZ######")


print(strip_accents("SAINT NICOLE-LA-FORÊT"))
print(random_date(), "|", random_city(), "|", random_card(), "|", random_ref())

SAINT NICOLE-LA-FORET
20/06 | PARIS 11 | CARTE X6977 | FR25ZZZ534192


In [6]:
# One label builder per payment type 

def card_label(merchant):
    date, city, card = random_date(), random_city(), random_card()
    templates = [
        f"CB {merchant} {date}",
        f"CB {merchant} {city} {date}",
        f"CARTE {date} {merchant} {city}",
        f"CB {merchant} {date} {card}",
        f"{card} {date} {merchant}",
        f"PAIEMENT PAR CARTE {merchant} {date}",
        f"PAIEMENT CB {date} {city} {merchant}",
    ]
    return random.choice(templates)


def direct_debit_label(merchant):
    ref = random_ref()
    templates = [
        f"PRLV SEPA {merchant}",
        f"PRLV SEPA {merchant} {ref}",
        f"PRELEVEMENT {merchant} REF {ref}",
        f"PRLV {merchant} {random_date()}",
    ]
    return random.choice(templates)


def transfer_out_label(recipient):
    templates = [
        f"VIR SEPA EMIS VERS {recipient}",
        f"VIR INST VERS {recipient}",
        f"VIREMENT EN FAVEUR DE {recipient}",
        f"VIR {recipient} {random_date()}",
    ]
    return random.choice(templates)


def transfer_in_label(sender):
    templates = [
        f"VIR SEPA RECU DE {sender}",
        f"VIR INST RECU {sender}",
        f"VIREMENT DE {sender}",
        f"VIR {sender} REF {random_ref()}",
    ]
    return random.choice(templates)


def withdrawal_label(bank):
    date, city = random_date(), random_city()
    templates = [
        f"RETRAIT DAB {date} {city}",
        f"RETRAIT DAB {bank} {city} {date}",
        f"RET DAB {date} {city} {random_card()}",
        f"RETRAIT {bank} {date}",
    ]
    return random.choice(templates)


def fee_label(fee):
    month = fake.date_between(start_date="-1y", end_date="today").strftime("%m/%Y")
    templates = [
        fee,
        f"{fee} {random_date()}",
        f"{fee} {month}",
    ]
    return random.choice(templates)


LABEL_BUILDERS = {
    "card": card_label,
    "debit": direct_debit_label,
    "transfer_out": transfer_out_label,
    "transfer_in": transfer_in_label,
    "withdrawal": withdrawal_label,
    "fee": fee_label,
}

In [7]:
# The full catalog 

EMPLOYERS = [strip_accents(fake.company().upper()) for _ in range(15)]
PEOPLE = [strip_accents(f"{fake.last_name()} {fake.first_name()}".upper()) for _ in range(40)]

CATALOG = {
    "Groceries": {
        "card": [
            "CARREFOUR", "CARREFOUR MARKET", "CARREFOUR CITY", "LIDL", "ALDI",
            "AUCHAN", "E.LECLERC", "LECLERC DRIVE", "MONOPRIX", "FRANPRIX",
            "INTERMARCHE", "SUPER U", "PICARD", "BIOCOOP", "NATURALIA",
            "GRAND FRAIS", "CASINO SHOP", "G20", "LA VIE CLAIRE",
        ],
    },
    "Restaurants": {
        "card": [
            "MCDONALDS", "BURGER KING", "KFC", "QUICK", "STARBUCKS", "COLUMBUS CAFE",
            "PAUL", "BRIOCHE DOREE", "DOMINOS PIZZA", "SUSHI SHOP", "O TACOS",
            "FIVE GUYS", "BUFFALO GRILL", "HIPPOPOTAMUS", "DELIVEROO", "UBER EATS",
            "JUST EAT", "SUMUP *CAFE DES AMIS", "ZETTLE *FOOD TRUCK",
            "BRASSERIE DU MARCHE", "CAFE DE LA GARE",
        ],
    },
    "Transport": {
        "card": [
            "SNCF CONNECT", "SNCF INTERNET", "OUIGO", "RATP", "UBER *TRIP", "BOLT.EU",
            "HEETCH", "BLABLACAR", "LIME", "DOTT", "TOTALENERGIES", "ESSO", "SHELL",
            "BP", "AVIA", "INDIGO PARKING", "SAEMES", "SANEF", "VINCI AUTOROUTES",
        ],
        "debit": ["NAVIGO", "COMUTITRES", "VELIB METROPOLE"],
    },
    "Subscriptions": {
        "card": [
            "NETFLIX.COM", "SPOTIFY", "DISNEY PLUS", "AMAZON PRIME", "APPLE.COM/BILL",
            "YOUTUBE PREMIUM", "GOOGLE *GOOGLE ONE", "MICROSOFT*365",
            "OPENAI *CHATGPT", "ADOBE", "AUDIBLE",
        ],
        "debit": ["CANAL+", "DEEZER", "LE MONDE", "L EQUIPE", "DAZN", "BEIN SPORTS"],
    },
    "Shopping": {
        "card": [
            "AMAZON", "AMAZON MKTPLACE", "FNAC", "DARTY", "BOULANGER", "DECATHLON",
            "ZARA", "H&M", "UNIQLO", "KIABI", "PRIMARK", "IKEA", "LEROY MERLIN",
            "CASTORAMA", "SEPHORA", "NOCIBE", "ACTION", "CDISCOUNT", "ZALANDO",
            "VINTED", "LEBONCOIN", "APPLE STORE", "GIFI",
        ],
    },
    "Housing & Utilities": {
        "debit": [
            "EDF", "ENGIE", "TOTALENERGIES ELECTRICITE GAZ", "FREE MOBILE",
            "FREE TELECOM", "ORANGE", "SFR", "BOUYGUES TELECOM", "SOSH", "RED BY SFR",
            "VEOLIA EAU", "SUEZ EAU", "MAIF", "MACIF", "AXA FRANCE",
            "DGFIP TAXE FONCIERE", "FONCIA",
        ],
        "transfer_out": ["LOYER", "AGENCE FONCIA", "NEXITY", "SCI LES LILAS", "CITYA"],
    },
    "Health": {
        "card": [
            "PHARMACIE DU CENTRE", "PHARMACIE DE LA GARE", "PHARMACIE LAFAYETTE",
            "GRANDE PHARMACIE", "DOCTOLIB*DR LEROY", "CABINET DENTAIRE",
            "CERBALLIANCE", "OPTIC 2000", "AFFLELOU", "CABINET KINE",
        ],
        "debit": ["MUTUELLE HARMONIE", "MGEN", "ALAN", "MALAKOFF HUMANIS"],
    },
    "Leisure & Travel": {
        "card": [
            "UGC", "PATHE", "MK2", "FNAC SPECTACLES", "TICKETMASTER", "AIRBNB",
            "BOOKING.COM", "AIR FRANCE", "EASYJET", "RYANAIR", "TRANSAVIA", "STEAM",
            "PLAYSTATION NETWORK", "NINTENDO", "DISNEYLAND PARIS", "PARC ASTERIX",
            "MUSEE DU LOUVRE",
        ],
        "debit": ["BASIC FIT", "FITNESS PARK", "NEONESS", "ON AIR FITNESS"],
    },
    "Income": {
        "transfer_in": EMPLOYERS + [
            "CAF", "CAF DE PARIS", "CPAM", "CPAM PARIS", "FRANCE TRAVAIL",
            "DGFIP REMBOURSEMENT IMPOT",
        ],
    },
    "Transfers": {
        "transfer_out": PEOPLE[:25] + [
            "LYDIA", "LIVRET A", "COMPTE EPARGNE", "WISE", "REVOLUT",
        ],
        "transfer_in": PEOPLE[25:] + ["LYDIA"],
    },
    "Cash Withdrawal": {
        "withdrawal": [
            "BNP PARIBAS", "SOCIETE GENERALE", "CREDIT AGRICOLE", "LCL", "CIC",
            "CREDIT MUTUEL", "LA BANQUE POSTALE", "CAISSE D EPARGNE",
            "BANQUE POPULAIRE", "HSBC",
        ],
    },
    "Bank Fees": {
        "fee": [
            "FRAIS TENUE DE COMPTE", "COTISATION CARTE VISA", "COTIS CARTE GOLD MC",
            "COMMISSION D INTERVENTION", "FRAIS REJET PRLV", "AGIOS",
            "INTERETS DEBITEURS", "FRAIS PAIEMENT HORS ZONE EURO",
            "COTIS OFFRE GROUPEE", "FRAIS ENVOI CHEQUIER",
        ],
    },
}

total = sum(len(merchants) for groups in CATALOG.values() for merchants in groups.values())
print(f"{len(CATALOG)} categories, {total} merchants")

12 categories, 246 merchants


In [8]:
# Test the generator 

def generate_label(category):
    payment_type = random.choice(list(CATALOG[category]))
    merchant = random.choice(CATALOG[category][payment_type])
    return LABEL_BUILDERS[payment_type](merchant)


for category in CATALOG:
    print(f"{category:<22}{generate_label(category)}")

Groceries             CB G20 ROUEN 27/01
Restaurants           PAIEMENT CB 20/09 MONTREUIL ZETTLE *FOOD TRUCK
Transport             CARTE 07/01 BOLT.EU TOULON
Subscriptions         PRLV SEPA DEEZER FR34ZZZ657871
Shopping              CB DARTY 12/01 CARTE X2049
Housing & Utilities   PRELEVEMENT SUEZ EAU REF FR50ZZZ983930
Health                PRLV MUTUELLE HARMONIE 02/02
Leisure & Travel      CARTE X8953 03/07 PLAYSTATION NETWORK
Income                VIREMENT DE DGFIP REMBOURSEMENT IMPOT
Transfers             VIR SEPA EMIS VERS RAYNAUD LOUIS
Cash Withdrawal       RETRAIT DAB 24/03 NANTERRE
Bank Fees             COTISATION CARTE VISA 23/11


In [9]:
# Generate thousands of labels 

import pandas as pd

N_PER_CATEGORY = 500


def generate_row(category):
    payment_type = random.choice(list(CATALOG[category]))
    merchant = random.choice(CATALOG[category][payment_type])
    label = LABEL_BUILDERS[payment_type](merchant)
    return {
        "label": label,
        "category": category,
        "merchant": merchant,
        "payment_type": payment_type,
    }


rows = [generate_row(category) for category in CATALOG for _ in range(N_PER_CATEGORY)]
df = pd.DataFrame(rows)

print(df.shape)
df.sample(10, random_state=42)

(6000, 4)


,label,category,merchant,payment_type
1782,CB DISNEY PLUS LA DEFENSE 21/11,Subscriptions,DISNEY PLUS,card
3917,PRLV SEPA ON AIR FITNESS FR46ZZZ943460,Leisure & Travel,ON AIR FITNESS,debit
221,PAIEMENT PAR CARTE CARREFOUR 08/09,Groceries,CARREFOUR,card
2135,CARTE 19/12 IKEA ROUEN,Shopping,IKEA,card
5224,RETRAIT BNP PARIBAS 24/01,Cash Withdrawal,BNP PARIBAS,withdrawal
1168,CB VINCI AUTOROUTES 16/05,Transport,VINCI AUTOROUTES,card
879,PAIEMENT PAR CARTE MCDONALDS 26/09,Restaurants,MCDONALDS,card
156,CB GRAND FRAIS CRETEIL 21/08,Groceries,GRAND FRAIS,card
1657,PRLV SEPA L EQUIPE,Subscriptions,L EQUIPE,debit
323,CB GRAND FRAIS 21/06 CARTE X9042,Groceries,GRAND FRAIS,card


In [10]:
# Check duplicates and conflicts 

n_duplicates = df.duplicated(subset="label").sum()
print(f"Duplicate labels: {n_duplicates} ({n_duplicates / len(df):.1%})")

conflicts = df.groupby("label")["category"].nunique()
print(f"Labels assigned to more than one category: {(conflicts > 1).sum()}")

df = df.drop_duplicates(subset="label").reset_index(drop=True)
print(f"\nRows after removing duplicates: {len(df)}\n")
print(df["category"].value_counts())

Duplicate labels: 1272 (21.2%)
Labels assigned to more than one category: 0

Rows after removing duplicates: 4728

category
Restaurants            500
Groceries              499
Shopping               499
Cash Withdrawal        495
Leisure & Travel       453
Health                 444
Transport              439
Subscriptions          434
Bank Fees              270
Housing & Utilities    261
Transfers              237
Income                 197
Name: count, dtype: int64


In [11]:
# How many different merchants per category 

df.groupby("category")["merchant"].nunique().sort_values()

category
Bank Fees              10
Cash Withdrawal        10
Health                 14
Subscriptions          17
Groceries              19
Income                 21
Restaurants            21
Leisure & Travel       21
Housing & Utilities    22
Transport              22
Shopping               23
Transfers              45
Name: merchant, dtype: int64

In [12]:
# Save the dataset

from pathlib import Path

output_path = Path("../data/raw/transactions_synthetic.csv")
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")

Saved 4728 rows to ..\data\raw\transactions_synthetic.csv
